<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-23</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>프롬프팅 기법(Prompting Techniques)</strong>에 대해 학습합니다.</br></br>
>Zero-shot, Few-shot, Chain-of-Thought 기법과 생성 파라미터를 학습해봅시다.

</br>

<div style="text-align:center">

</div>

In [3]:
# TODO 0: LLM 호출에 필요한 라이브러리를 불러오고, LLM 객체를 생성해봅시다.

from dotenv import load_dotenv
from langchain_upstage import ChatUpstage

load_dotenv()

llm = ChatUpstage(model="solar-pro3", temperature=0)

</br>

# 프롬프팅 (Prompting)
> LLM에게 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">적절한 지시문을 제공</mark>하여 원하는 출력을 얻는 기법입니다.

> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">동일한 모델</mark>에 동일한 질문을 하더라도, 프롬프트의 표현 방식에 따라 답변 품질이 크게 달라집니다.</br></br>
> 프롬프트 엔지니어링이란 LLM이 원하는 형식, 수준, 내용으로 응답하도록 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">입력(프롬프트)을 체계적으로 설계</mark>하는 기술이며, 다음 토큰을 예측하는 LLM의 특성상 모델 재학습 없이 즉시 적용할 수 있어 실무에서 가장 먼저 시도하는 성능 개선 방법입니다.</br></br>
> 같은 질문도 "분류하세요" vs "긍정/부정 중 하나로만 답하세요"는 결과가 다르며, 예시를 제공하면 모델이 원하는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">출력 패턴을 학습</mark>합니다(Few-shot).</br></br>
> 단계별 사고를 유도하면 수학, 논리 문제의 정확도가 올라가고(Chain-of-Thought), temperature 등 파라미터 조절로 창의성과 일관성 사이의 균형을 잡을 수 있습니다.

## Zero-shot Prompting
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">예시 없이</mark> 태스크 설명만으로 답변을 요청합니다.

In [4]:
# TODO 1: Zero-shot 프롬프트를 작성하여 "이 영화 정말 재미있어요!"의 감성을 '긍정' 또는 '부정'으로 분류하고, LLM을 호출하여 결과를 출력해봅시다.

prompt = "다음 문장의 감성을 '긍정' 또는 '부정'으로 분류하세요: 이 영화 정말 재미있어요!"
response = llm.invoke(prompt)
print(response.content)

긍정


</br>

## Few-shot Prompting
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">몇 개의 예시를 포함</mark>하여 모델이 패턴을 학습하게 합니다.

In [5]:
# TODO 2: 긍정/부정 예시 2개를 포함한 Few-shot 프롬프트를 작성하고, "가격 대비 품질이 괜찮네요."의 감성을 분류하도록 LLM을 호출하여 결과를 출력해봅시다.

prompt = """다음 문장의 감성을 분류하세요.

문장: 정말 최고의 서비스였습니다.
감성: 긍정

문장: 배송이 너무 늦어서 실망했어요.
감성: 부정

문장: 가격 대비 품질이 괜찮네요.
감성:
"""

response = llm.invoke(prompt)
print(response.content)

문장: 가격 대비 품질이 괜찮네요.  
감성: 긍정


💡Few-shot 예시 선택
> 예시는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다양하고 대표적인 사례</mark>를 포함해야 합니다.</br>
> 2~5개 정도가 일반적이며, 너무 많으면 토큰 낭비가 됩니다.

</br>

## Chain-of-Thought (CoT)
> 모델에게 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단계별로 사고 과정을 설명</mark>하게 하여 추론 정확도를 높입니다.

In [6]:
# TODO 3: 사과 재고 계산 문제를 단계별로 풀어가는 Chain-of-Thought 프롬프트를 작성하고, LLM을 호출하여 결과를 출력해봅시다.

prompt = """Q: 상점에 사과가 23개 있었고, 12개를 팔고 15개를 새로 받았습니다.
현재 사과는 몇 개인가요?

A: 단계별로 생각해봅시다.
1. 처음 사과: 23개
2. 판매 후: 23 - 12 = 11개
3. 입고 후: 11 + 15 = 26개
따라서 현재 사과는 26개입니다."""

response = llm.invoke(prompt)
print(response.content)

현재 사과는 **26개**입니다.  

**단계별 계산**  
1. 처음 사과: 23개  
2. 판매 후: 23 − 12 = 11개  
3. 새로 입고 후: 11 + 15 = 26개  

따라서 최종적으로 26개의 사과가 남아 있습니다.


💡CoT의 효과
> 수학 문제, 논리 추론, 코드 생성 등 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">복잡한 태스크에서 성능이 크게 향상</mark>됩니다.</br>
> "Let's think step by step"만 추가해도 효과가 있습니다.

</br>

# 생성 파라미터 (Generation Parameters)

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
      <th style="text-align:center">범위</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>temperature</code></td><td>출력 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다양성</mark> 조절</td><td style="text-align:center">0.0 ~ 2.0</td></tr>
    <tr><td style="text-align:center"><code>top_p</code></td><td>누적 확률 기반 샘플링</td><td style="text-align:center">0.0 ~ 1.0</td></tr>
    <tr><td style="text-align:center"><code>max_tokens</code></td><td>최대 출력 토큰 수</td><td style="text-align:center">양의 정수</td></tr>
  </tbody>
</table>

In [8]:
# TODO 4: Upstage API 클라이언트를 생성하고, "AI의 미래를 한 문장으로"라는 요청을 낮은 temperature로 호출하여 결과를 출력해봅시다.

import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("UPSTAGE_API_KEY"),
    base_url="https://api.upstage.ai/v1"  # Upstage API 엔드포인트
)
response = client.chat.completions.create(
    model="solar-pro3",
    messages=[{"role": "user", "content": "AI의 미래를 한 문장으로"}],
    temperature=0.0,    # 결정적 출력
    max_tokens=500
)
print(response.choices[0].message.content)

AI는 인간의 창의와 협업을 확장하며, 스스로 학습하고 진화하는 지능으로 우리 삶의 모든 영역에 스며들어 새로운 가능성을 열어갈 것이다.


💡temperature 설정 가이드
> `0.0`: 정확한 답변 (분류, 추출) — 항상 같은 결과</br>
> `0.3~0.7`: 균형 (일반 대화, 요약)</br>
> `1.0+`: 창의적 출력 (스토리 작성, 브레인스토밍)